# 🎮 LivreGames 3D Cloud Gaming (Google Colab GPU)

Este notebook transforma sua instância do **Google Colab com GPU Tesla T4 (16 GB)** em um servidor de **Cloud Gaming 3D** de alta performance.

- ⚡ **Renderização na Nuvem:** O jogo 3D de PSP roda na GPU Nvidia da Google.
- 📱 **Zero Download no Celular:** Nenhum arquivo pesado (ISO/CSO) é baixado no celular.
- 🚀 **60 FPS & Baixa Latência:** Vídeo transmitido em tempo real via WebSocket.
- 🕹️ **Controles Touch Integrados:** D-pad, botões ▲ ■ ● ✕, L/R e Start/Select na tela do celular.

In [ ]:
# 1. Verificar a GPU ativa no Colab
!nvidia-smi

In [ ]:
# @title 🚀 2. INICIAR SERVIDOR CLOUD GAMING { run: "auto" }
# @markdown Escolha o jogo e clique no botão **Play (Executar)** ao lado:

JOGO = "Dragon Ball Z: Shin Budokai" #@param ["Dragon Ball Z: Shin Budokai", "Bomberman Panic Bomber"]
RESOLUCAO = "960x544 (2x PSP HD)" #@param ["960x544 (2x PSP HD)", "1280x720 (720p HD)", "480x272 (1x PSP)"]

GAMES = {
    "Dragon Ball Z: Shin Budokai": {
        "url": "https://pub-425fc23f99ae41f4bf6e29d5f0d07e12.r2.dev/games/dbz_shin_budokai.cso",
        "file": "dbz_shin_budokai.cso",
        "title": "Dragon Ball Z: Shin Budokai"
    },
    "Bomberman Panic Bomber": {
        "url": "https://pub-425fc23f99ae41f4bf6e29d5f0d07e12.r2.dev/games/bomberman.cso",
        "file": "bomberman.cso",
        "title": "Bomberman Panic Bomber"
    }
}

RES_MAP = {
    "960x544 (2x PSP HD)": (960, 544, 2),
    "1280x720 (720p HD)": (1280, 720, 3),
    "480x272 (1x PSP)": (480, 272, 1)
}

WIDTH, HEIGHT, PSP_SCALE = RES_MAP.get(RESOLUCAO, (960, 544, 2))
SELECTED_GAME = GAMES.get(JOGO, GAMES["Dragon Ball Z: Shin Budokai"])

import os, sys, time, json, re, subprocess, asyncio
from pathlib import Path

# 1. Instalar dependências silenciosamente
print("📦 [1/6] Configurando ambiente Linux (Xvfb, FFmpeg, xdotool, PulseAudio)...")
os.system("apt-get update -qq > /dev/null 2>&1")
os.system("apt-get install -y -qq xvfb xdotool ffmpeg pulseaudio libgl1-mesa-dri libsdl2-2.0-0 > /dev/null 2>&1")
os.system("pip install -q aiohttp websockets qrcode[pil] > /dev/null 2>&1")

# 2. Baixar binários essenciais
if not os.path.exists("jsmpeg.min.js"):
    os.system("wget -q -nc https://raw.githubusercontent.com/phoboslab/jsmpeg/master/jsmpeg.min.js -O jsmpeg.min.js")
if not os.path.exists("cloudflared"):
    os.system("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
    os.system("chmod +x cloudflared")

# 3. Preparar PPSSPP
print("🎮 [2/6] Preparando emulador PPSSPP...")
if not os.path.exists("squashfs-root/AppRun"):
    os.system("wget -q -nc https://github.com/hrydgard/ppsspp/releases/download/v1.18.1/PPSSPP-v1.18.1-anylinux-x86_64.AppImage -O ppsspp.AppImage")
    os.system("chmod +x ppsspp.AppImage")
    os.system("./ppsspp.AppImage --appimage-extract > /dev/null 2>&1")

config_dir = Path.home() / ".config" / "ppsspp" / "PSP" / "SYSTEM"
config_dir.mkdir(parents=True, exist_ok=True)
(config_dir / "ppsspp.ini").write_text(f"""[General]
FirstRun = False
AutoLoadSaveState = 0
EnableCheats = False
[Graphics]
ShowFPSCounter = 0
InternalResolution = {PSP_SCALE}
WindowWidth = {WIDTH}
WindowHeight = {HEIGHT}
FullScreen = True
VSyncInterval = False
RenderingMode = 1
SoftwareRendering = False
HardwareTransform = True
SoftwareSkinning = True
[Audio]
Enable = True
AudioBackend = 0
AudioVolume = 10
[Control]
ShowTouchControls = False
""")

# 4. Baixar ROM
print(f"📥 [3/6] Baixando jogo {SELECTED_GAME['title']} (conexão 1Gbps do Colab)...")
games_dir = Path("/content/games")
games_dir.mkdir(parents=True, exist_ok=True)
game_target = games_dir / SELECTED_GAME['file']
if not game_target.exists() or game_target.stat().st_size < 1000000:
    os.system(f"wget -q --show-progress -O {game_target} {SELECTED_GAME['url']}")

# 5. Display Virtual & Som
print("🖥️ [4/6] Iniciando display virtual (:1) e subsistema de áudio...")
os.system("pkill -9 Xvfb > /dev/null 2>&1")
os.system(f"Xvfb :1 -screen 0 {WIDTH}x{HEIGHT}x24 +extension GLX +render -noreset &")
time.sleep(1)
os.system("pkill -9 pulseaudio > /dev/null 2>&1")
os.system("pulseaudio --system --disallow-exit --disable-shm --exit-idle-time=-1 --daemonize=yes > /dev/null 2>&1")

# 6. Iniciar Servidor Web, Game & Stream
print("🚀 [5/6] Subindo servidor WebRTC/WebSocket e codificador FFmpeg 60 FPS...")
os.system("pkill -9 -f colab_cloud_gaming.py > /dev/null 2>&1")
os.system("pkill -9 AppRun > /dev/null 2>&1")
os.system("pkill -9 ffmpeg > /dev/null 2>&1")
os.system("pkill -9 cloudflared > /dev/null 2>&1")

# Baixar script oficial atualizado diretamente do CDN R2
os.system("wget -q -O /content/colab_cloud_gaming.py https://pub-425fc23f99ae41f4bf6e29d5f0d07e12.r2.dev/cloud-gaming/colab_cloud_gaming.py || true")

import colab_cloud_gaming
colab_cloud_gaming.WIDTH = WIDTH
colab_cloud_gaming.HEIGHT = HEIGHT
colab_cloud_gaming.DEFAULT_GAME_URL = SELECTED_GAME['url']
colab_cloud_gaming.GAME_NAME = SELECTED_GAME['title']

colab_cloud_gaming.main()
